In [3]:
import os
import numpy as np
from PIL import Image

### Image Preprocessing

1. Grayscale
2. Thresholding
3. Cropping/Bounding Box

In [47]:
def rgb2gray(rgb):
    return np.dot(rgb[..., :3], [0.2989, 0.5870, 0.1140]).astype(np.uint8)

In [49]:
chulpi_input_path = 'dataset/Gaussian Naive Bayes/Zea_mays_Chulpi_Cancha/'
chulpi_output_path = 'dataset/Gaussian Naive Bayes/output/Zea_mays_Chulpi_Cancha/'

indurata_input_path = 'dataset/Gaussian Naive Bayes/Zea_mays_Indurata/'
indurata_output_path = 'dataset/Gaussian Naive Bayes/output/Zea_mays_Indurata/'

rugosa_input_path = 'dataset/Gaussian Naive Bayes/Zea_mays_Rugosa/'
rugosa_output_path = 'dataset/Gaussian Naive Bayes/output/Zea_mays_Rugosa/'

### Make output file

In [50]:
def mkdir(output_path):
    if not os.path.exists(output_path):
        os.makedirs(output_path)

In [51]:
mkdir(chulpi_output_path)
mkdir(indurata_output_path)
mkdir(rugosa_output_path)

### Import IMG Dataset

In [52]:
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp")

In [55]:
def gen_gray(input_path, output_path):
    image_files = [f for f in os.listdir(input_path) if f.lower().endswith(valid_extensions)]
    print(f"The total img: {len(image_files)}")

    image_files = input_path + np.asarray(image_files)
    color_img = [Image.open(str(i)).convert('RGB') for i in image_files]
    img_array = np.asarray(color_img)

    print(f'The img shape: {img_array.shape}')

    # Parse color img into gray color
    for idx, img in enumerate(img_array):
        img_height_size = len(img)
        img_width_size = len(img[0])
        gray_array = np.zeros((img_height_size, img_width_size), dtype=np.uint8)
        for i in range(img_height_size):
            for j in range(img_width_size):
                gray_array[i, j] = int(rgb2gray(img[i, j]))
        img_gray = Image.fromarray(gray_array)
        img_gray.save(f'{output_path}{idx}.jpg')


In [56]:
gen_gray(chulpi_input_path, chulpi_output_path)
gen_gray(indurata_input_path, indurata_output_path)
gen_gray(rugosa_input_path, rugosa_output_path)

The total img: 350
The img shape: (350, 400, 400, 3)
The total img: 350
The img shape: (350, 400, 400, 3)
The total img: 350
The img shape: (350, 400, 400, 3)


### Binary Thresholding

Binary thresholding converts a grayscale image into a binary image
containing only black (0) and white (255).

For each pixel:

$$
g(i,j) =
\begin{cases}
255 & \text{if } I(i,j) \geq T \\
0 & \text{if } I(i,j) < T
\end{cases}
$$

**Where:**

- $I(i,j)$ → original pixel intensity
- $T$ → threshold
- $g(i,j)$ → output pixel
- $255$ → white
- $0$ → black

This allows bright objects, such as corn kernels, to be separated
from the darker background.

In [57]:
THREADHOLD_VALUE = 30

In [58]:
chulpi_input_path = 'dataset/Gaussian Naive Bayes/output/Zea_mays_Chulpi_Cancha/'
chulpi_output_path = 'dataset/Gaussian Naive Bayes/binary/Zea_mays_Chulpi_Cancha/'

indurata_input_path = 'dataset/Gaussian Naive Bayes/output/Zea_mays_Indurata/'
indurata_output_path = 'dataset/Gaussian Naive Bayes/binary/Zea_mays_Indurata/'

rugosa_input_path = 'dataset/Gaussian Naive Bayes/output/Zea_mays_Rugosa/'
rugosa_output_path = 'dataset/Gaussian Naive Bayes/binary/Zea_mays_Rugosa/'

In [59]:
def gen_binary(input_path, output_path):
    image_files = [f for f in os.listdir(input_path) if f.lower().endswith(valid_extensions)]
    print(f"The total img: {len(image_files)}")

    image_files = input_path + np.asarray(image_files)
    gray_img = [Image.open(str(i)).convert('L') for i in image_files]
    img_array = np.asarray(gray_img)

    print(f'The img shape: {img_array.shape}')

    # Parse color img into gray color
    for idx, img in enumerate(img_array):
        img_height_size = len(img)
        img_width_size = len(img[0])
        binary_array = np.zeros((img_height_size, img_width_size), dtype=np.uint8)
        for i in range(img_height_size):
            for j in range(img_width_size):
                binary_array[i, j] = 255 if (img[i, j] >= THREADHOLD_VALUE) else 0
        img_binary = Image.fromarray(binary_array)
        img_binary.save(f'{output_path}{idx}.jpg')

In [61]:
mkdir(chulpi_output_path)
mkdir(indurata_output_path)
mkdir(rugosa_output_path)

In [62]:
gen_binary(chulpi_input_path, chulpi_output_path)
gen_binary(indurata_input_path, indurata_output_path)
gen_binary(rugosa_input_path, rugosa_output_path)

The total img: 350
The img shape: (350, 400, 400)
The total img: 350
The img shape: (350, 400, 400)
The total img: 350
The img shape: (350, 400, 400)


## Cropping / Bounding Box

### 1. Mathematical Concept

After obtaining a binary image, the next step is to determine the location of the object, such as a corn kernel, in order to limit the region used for further processing.

The computer scans the binary image matrix and finds the four boundaries of the white object (pixel value = 255):

- **Top**: the smallest row index $i$ where a white pixel appears.
- **Bottom**: the largest row index $i$ where a white pixel appears.
- **Left**: the smallest column index $j$ where a white pixel appears.
- **Right**: the largest column index $j$ where a white pixel appears.

These four coordinates define the **Bounding Box** of the object.

Mathematically:

$$
\begin{aligned}
Top &= \min\{i \mid I(i,j)=255\} \\
Bottom &= \max\{i \mid I(i,j)=255\} \\
Left &= \min\{j \mid I(i,j)=255\} \\
Right &= \max\{j \mid I(i,j)=255\}
\end{aligned}
$$

Once the four boundaries are determined, **matrix slicing** can be used to extract the region between them:

$$
I_{crop} = I[Top:Bottom+1,\; Left:Right+1]
$$

The result is a smaller image containing only the region occupied by the corn kernel, which reduces the area that needs to be processed in subsequent steps.

In [79]:
def gen_cropped(input_path, output_path):
    image_files = [f for f in os.listdir(input_path) if f.lower().endswith(valid_extensions)]
    print(f"The total img: {len(image_files)}")

    image_files = input_path + np.asarray(image_files)
    binary_img = [Image.open(str(i)).convert('1') for i in image_files]
    img_array = np.asarray(binary_img)

    print(f'The img shape: {img_array.shape}')

    # Parse color img into gray color
    for idx, img in enumerate(img_array):
        img_height_size = len(img)
        img_width_size = len(img[0])
        top = img_height_size
        bottom = 0
        left = img_width_size
        right = 0
        for i in range(img_height_size):
            for j in range(img_width_size):
                if img[i, j] == 255:
                    print(f"Is 255") 
                    if i < top: top = i
                    if i > bottom: bottom = i
                    if j < left: left = j
                    if j > right: right = j
        cropped_array = img[top:bottom+1, left:right+1]
        img_cropped = Image.fromarray(cropped_array)
        img_cropped.save(f'{output_path}{idx}.jpg')

In [70]:
chulpi_input_path = 'dataset/Gaussian Naive Bayes/binary/Zea_mays_Chulpi_Cancha/'
chulpi_output_path = 'dataset/Gaussian Naive Bayes/cropped/Zea_mays_Chulpi_Cancha/'

indurata_input_path = 'dataset/Gaussian Naive Bayes/binary/Zea_mays_Indurata/'
indurata_output_path = 'dataset/Gaussian Naive Bayes/cropped/Zea_mays_Indurata/'

rugosa_input_path = 'dataset/Gaussian Naive Bayes/binary/Zea_mays_Rugosa/'
rugosa_output_path = 'dataset/Gaussian Naive Bayes/cropped/Zea_mays_Rugosa/'

In [71]:
mkdir(chulpi_output_path)
mkdir(indurata_output_path)
mkdir(rugosa_output_path)

In [80]:
gen_cropped(chulpi_input_path, chulpi_output_path)
gen_cropped(indurata_input_path, indurata_output_path)
gen_cropped(rugosa_input_path, rugosa_output_path)

The total img: 350
The img shape: (350, 400, 400)


ValueError: cannot write empty image as JPEG

### Morphological Features